In [1]:
# ==========================================
# Imports
# ==========================================

import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, classification_report
from anova_module import ModelAnalysis

In [2]:
# ==========================================
# 1. LOADING AND PREPARATION (Direct URL)
# ==========================================
print("Downloading and reading data into memory...")
# Reading directly from UCI URLs
train_df = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/poker/poker-hand-training-true.data", header=None)
test_df = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/poker/poker-hand-testing.data", header=None)

# Splitting X/y
X_train = train_df.iloc[:, :-1].values
y_train = train_df.iloc[:, -1].values
X_test = test_df.iloc[:, :-1].values
y_test = test_df.iloc[:, -1].values

# Adjusting indices to start at 0 (required for PyTorch Embedding)
# S (Even columns): 1..4 -> 0..3 | C (Odd columns): 1..13 -> 0..12
X_train[:, 0::2] -= 1; X_train[:, 1::2] -= 1
X_test[:, 0::2] -= 1;  X_test[:, 1::2] -= 1

# Conversion to Tensors and DataLoaders
BATCH_SIZE = 256
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_loader = DataLoader(TensorDataset(torch.LongTensor(X_train), torch.LongTensor(y_train)), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(torch.LongTensor(X_test), torch.LongTensor(y_test)), batch_size=BATCH_SIZE)

# ==========================================
# 2. TRANSFORMER MODEL
# ==========================================
class TabularTransformer(nn.Module):
    def __init__(self, embed_dim=32, num_heads=4, ff_dim=64):
        super().__init__()
        self.suit_embed = nn.Embedding(4, embed_dim)   # 4 suits
        self.rank_embed = nn.Embedding(13, embed_dim)  # 13 ranks
        
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)
        
        # 10 features * embed_dim input for the classifier
        self.classifier = nn.Sequential(
            nn.Linear(10 * embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, 10) # 10 poker hand classes
        )
        
    def forward(self, x):
        # Separation and embedding
        s_emb = self.suit_embed(x[:, 0::2]) # (Batch, 5, Dim)
        r_emb = self.rank_embed(x[:, 1::2]) # (Batch, 5, Dim)
        
        # Interleaving to reconstruct the sequence [S1, C1, S2, C2...]
        x_emb = torch.zeros(x.shape[0], 10, s_emb.shape[-1], device=x.device)
        x_emb[:, 0::2] = s_emb
        x_emb[:, 1::2] = r_emb
        
        # Transformer -> Flatten -> MLP
        out = self.transformer(x_emb)
        return self.classifier(out.reshape(out.shape[0], -1))

model = TabularTransformer().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# ==========================================
# 3. TRAINING
# ==========================================
print(f"Starting training on {device}...")
model.train()
for epoch in range(10): # 10 Epochs are sufficient for testing
    total_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Average Loss = {total_loss/len(train_loader):.4f}")

# ==========================================
# 4. FINAL EVALUATION
# ==========================================
print("\n--- Test Set Results ---")
model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(labels.numpy())

print(f"Accuracy: {accuracy_score(all_targets, all_preds):.4f}")
print(classification_report(all_targets, all_preds, zero_division=0))

Starting training on cpu...
Epoch 1: Average Loss = 1.0617
Epoch 2: Average Loss = 0.9641
Epoch 3: Average Loss = 0.6143
Epoch 4: Average Loss = 0.3050
Epoch 5: Average Loss = 0.1807
Epoch 6: Average Loss = 0.1381
Epoch 7: Average Loss = 0.1227
Epoch 8: Average Loss = 0.1086
Epoch 9: Average Loss = 0.0990
Epoch 10: Average Loss = 0.0893

--- Test Set Results ---
Accuracy: 0.9843
              precision    recall  f1-score   support

           0       0.99      1.00      0.99    501209
           1       1.00      1.00      1.00    422498
           2       0.86      0.97      0.91     47622
           3       0.88      0.64      0.74     21121
           4       0.00      0.00      0.00      3885
           5       0.00      0.00      0.00      1996
           6       0.83      0.75      0.79      1424
           7       0.00      0.00      0.00       230
           8       0.00      0.00      0.00        12
           9       0.00      0.00      0.00         3

    accuracy          

In [ ]:
# ==========================================
# Function
# ==========================================

X_numpy = X_train
r , d = X_numpy.shape

# Function of interest (proba class 0)
def f_model(X_numpy):
    
    X_tensor = torch.LongTensor(X_numpy).to(device)
    
    model.eval()
    with torch.no_grad():
        logits = model(X_tensor)      
        probs = torch.softmax(logits, dim=1)
        
        predictions_class_1 = probs[:, 0]

    return predictions_class_1.cpu().numpy()

In [4]:
%%time
# =============================================
# Functional ANOVA Decomposition (MAIN EFFECTS)
# =============================================

A = ModelAnalysis(X_numpy , f_model , 0.305 , 1 , 1e-4) # percentage = 0.305 to have exactly all main effects
S , Matrix = A.functional_anova() # sets and f_A(X_A)
print(A.get_R2() , A.get_L2_Error() , A.get_L2_Error_rel())

Constructing Basis Matrix: 100%|██████████| 76/76 [00:00<00:00, 363.45it/s]


Computations complete. Results ready.
0.0029647047048213526 0.24470471391357562 0.4925174685588371
CPU times: user 3.86 s, sys: 723 ms, total: 4.58 s
Wall time: 1.77 s


In [5]:
%%time
# =============================================
# Functional ANOVA Decomposition
# =============================================

A = ModelAnalysis(X_numpy , f_model , 20 , 1 , 1e-4)
S , Matrix = A.functional_anova() # sets and f_A(X_A)
print(A.get_R2() , A.get_L2_Error() , A.get_L2_Error_rel())

Constructing Basis Matrix: 100%|██████████| 5001/5001 [11:07<00:00,  7.49it/s]


Computations complete. Results ready.
0.7870539727935567 0.052263843529388815 0.10519149999431907
CPU times: user 11min 41s, sys: 7min 52s, total: 19min 34s
Wall time: 11min 45s
